# Document type classification: fine-tune Qwen3.5-4B and Gemma 4 E4B

One notebook, two fine-tunes on the same 200-row golden dataset used for this task's GEPA prompt-optimization run, evaluated with the **same exact-match metric on the same held-out split**, so results are directly comparable across model families.

The teacher scores baked into the comparison cells come from the published run in this task's README (gpt-5.4-mini, 2026-07-10: 78.33 with the verbatim shipped paperless-gpt prompt, 81.67 GEPA-optimized). This notebook does NOT re-run GEPA -- only the two fine-tunes below are new measurements.

**Go criterion:** each fine-tuned SLM matches or beats gpt-5.4-mini's plain-prompt baseline (78.33).

Runtime: Colab **T4 (free)** works for each model in 4-bit; A100 (Colab Pro) is faster. Runtime -> Change runtime type -> GPU. Models are loaded and fine-tuned one at a time, with GPU memory freed between each, so both fit in one T4 session.

In [ ]:
# 1. Install (Colab)
# Pinned: unsloth/unsloth-zoo 2026.7.1 (released 2026-07-07) added a new XET-based
# downloader (unsloth_zoo/hf_xet_fallback.py) that fails on Colab with
# DownloadStallError even when HF_HUB_DISABLE_XET=1 is set. These are the last
# versions before that change -- the same ones the receipt-extraction notebooks
# ran successfully on.
!pip install -q unsloth==2026.6.9 unsloth-zoo==2026.6.7

In [ ]:
# 2. Fetch the same golden dataset (no upload needed) and replicate the GEPA run's exact split
import json
import random
import urllib.parse
import urllib.request
from collections import defaultdict

SEED = 42
ROW_COUNT = 200
DATASET = "anirudh1112/corrected-tobacco-dataset-with-ocr"
SPLITS = {"train": 2186, "validation": 272, "test": 279}
PAGE_SIZE = 100
MAX_CONTENT_CHARS = 4000
CLASS_NAMES = ["ADVE", "Email", "Form", "Letter", "Memo", "News", "Note", "Report", "Resume", "Scientific"]
PROMPT_TEMPLATE = """I will provide you with the content and the title of a document.
Your task is to select the most appropriate document type for the document from the list of available document types I will provide.
Only select a document type from the provided list. Respond only with the selected document type name, without any additional information.
If none of the available document types fit the document, respond with an empty string.
The content is likely in English.

The data will be provided using an XML-like format for clarity:

<available_document_types>
{available_document_types}
</available_document_types>

<title>
</title>

<content>
{content}
</content>

Please select the single most appropriate English document type from the list above that best categorizes this document.
Be selective and only choose a document type if it clearly matches the document's nature (e.g., Invoice, Contract, Receipt, Letter, etc.).
"""

def fetch_split(split, total_rows):
    rows = []
    for offset in range(0, total_rows, PAGE_SIZE):
        params = urllib.parse.urlencode({
            "dataset": DATASET,
            "config": "default",
            "split": split,
            "offset": offset,
            "length": min(PAGE_SIZE, total_rows - offset),
        })
        with urllib.request.urlopen(f"https://datasets-server.huggingface.co/rows?{params}", timeout=120) as resp:
            data = json.load(resp)
        rows.extend(item["row"] for item in data["rows"])
    return rows

def build_input(text):
    # ponytail: char cap approximates paperless-gpt token truncation for this notebook.
    return PROMPT_TEMPLATE.format(
        available_document_types=", ".join(CLASS_NAMES),
        content=text.strip()[:MAX_CONTENT_CHARS],
    )

source_rows = []
for split_name, total in SPLITS.items():
    source_rows.extend(fetch_split(split_name, total))

by_label = defaultdict(list)
for row in source_rows:
    label = row.get("label", "").strip()
    text = row.get("text", "").strip()
    if label in CLASS_NAMES and text:
        by_label[label].append(row)

rng = random.Random(SEED)
rows = []
for label in CLASS_NAMES:
    rows.extend(rng.sample(by_label[label], ROW_COUNT // len(CLASS_NAMES)))
rng.shuffle(rows)
rows = [{"text": build_input(row["text"]), "expected": row["label"].strip()} for row in rows]

random.Random(SEED).shuffle(rows)
split = int(len(rows) * 0.7)
trainset, devset = rows[:split], rows[split:]
print(f"train {len(trainset)}, held-out {len(devset)}")


In [ ]:
# 3. Same exact-match metric as the GEPA spike (--metric exact)
def exact_match(expected: str, actual: str) -> float:
    return 1.0 if actual.strip().casefold() == expected.strip().casefold() else 0.0


In [ ]:
# 4. Eval + train helpers, shared by both models below (identical logic,
# defined once rather than twice)
import re

def clean_label(s: str) -> str:
    s = re.sub(r"<think>.*?(</think>|$)", "", s, flags=re.DOTALL)
    s = re.sub(r"```.*?```", "", s, flags=re.DOTALL)
    # Strip wrapping quotes BEFORE the emptiness check: a completion that is
    # only quotes ('""') stripped to nothing crashed splitlines()[0] here.
    s = s.strip().strip('"').strip()
    return s.splitlines()[0].strip() if s else ""

def evaluate_slm(model, tokenizer, devset) -> float:
    from unsloth import FastLanguageModel
    FastLanguageModel.for_inference(model)
    scores = []
    for row in devset:
        messages = [{"role": "user", "content": row["text"]}]
        try:
            prompt = tokenizer.apply_chat_template(
                messages, add_generation_prompt=True, tokenize=False, enable_thinking=False
            )
        except TypeError:
            prompt = tokenizer.apply_chat_template(
                messages, add_generation_prompt=True, tokenize=False
            )
        # text= keyword, not a positional arg: multimodal processors may treat a positional arg as an image.
        inputs = tokenizer(text=prompt, return_tensors="pt").to(model.device)
        out = model.generate(**inputs, max_new_tokens=32, do_sample=False)
        completion = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        scores.append(exact_match(row["expected"], clean_label(completion)))
    return 100 * sum(scores) / len(scores)

def to_text(tokenizer, row):
    messages = [
        {"role": "user", "content": row["text"]},
        {"role": "assistant", "content": row["expected"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

def free_gpu_memory():
    import gc
    import torch
    gc.collect()
    torch.cuda.empty_cache()

results = {}  # each model's raw/fine-tuned scores land here as we go

## Qwen3.5-4B

`Qwen/Qwen3.5-4B`

In [ ]:
# 5. Load Qwen3.5-4B in 4-bit + LoRA
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    "Qwen/Qwen3.5-4B",
    max_seq_length=4096,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

In [ ]:
# 6. Pre-finetune baseline of the raw SLM on the held-out split
raw_slm_score = evaluate_slm(model, tokenizer, devset)
print(f"Qwen3.5-4B raw (no fine-tune): {raw_slm_score:.2f}")

In [ ]:
# 7. Build chat-format training data and fine-tune (SFT)
from datasets import Dataset
from trl import SFTConfig, SFTTrainer

train_ds = Dataset.from_list([to_text(tokenizer, r) for r in trainset])

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        logging_steps=5,
        output_dir="outputs-apprentice-qwen35-4b-lora-document-types",
        report_to="none",
    ),
)
trainer.train()

In [ ]:
# 8. Post-finetune eval on the SAME held-out split: the comparison that matters
finetuned_score = evaluate_slm(model, tokenizer, devset)
results["Qwen3.5-4B"] = {"raw": raw_slm_score, "finetuned": finetuned_score}

gpt54_baseline = 78.33  # gpt-5.4-mini, verbatim shipped paperless-gpt prompt (README, 2026-07-10)
gpt54_optimized = 81.67  # same run, GEPA-optimized

print("=" * 52)
print(f"gpt-5.4-mini baseline prompt : {gpt54_baseline:.2f}")
print(f"gpt-5.4-mini GEPA-optimized  : {gpt54_optimized:.2f}")
print(f"Qwen3.5-4B raw               : {raw_slm_score:.2f}")
print(f"Qwen3.5-4B fine-tuned        : {finetuned_score:.2f}")
print("=" * 52)
print("GO criterion: fine-tuned SLM >= gpt-5.4-mini baseline prompt score (78.33)")

In [ ]:
# 9. Persist the adapter (plan: artifacts go to a private HF Hub repo, not Drive),
# then free GPU memory before the next model
model.save_pretrained("apprentice-qwen35-4b-lora-document-types")
tokenizer.save_pretrained("apprentice-qwen35-4b-lora-document-types")
print("saved -> apprentice-qwen35-4b-lora-document-types/")

del model, tokenizer, trainer
free_gpu_memory()

In [ ]:
# 9b. Write the model card and push to your HF Hub repo (same card style as
# the published receipt-extraction adapters).
# Fill in the printed numbers from cell 8 before pushing. Never push a card
# with a number that was not actually printed by this run.
from huggingface_hub import login

login()  # paste a HF token with write access when prompted

HF_USERNAME = "YOUR_HF_USERNAME"
REPO_ID = f"{HF_USERNAME}/apprentice-qwen35-4b-lora-document-types"

model_card = "---\ndatasets: [anirudh1112/corrected-tobacco-dataset-with-ocr]\nbase_model: Qwen/Qwen3.5-4B\nlibrary_name: peft\nlicense: apache-2.0\ntags: [lora, unsloth, qwen, document-type-classification, apprentice, paperless-gpt]\n---\n\n# Apprentice Qwen3.5-4B LoRA (document type classification)\n\nLoRA adapter fine-tuned on 140 golden examples from a corrected Tobacco3482 OCR-text dataset to classify one document type from: ADVE, Email, Form, Letter, Memo, News, Note, Report, Resume, Scientific.\n\nThe input prompt is the verbatim shipped document-type prompt from icereed/paperless-gpt, filled with English, the allowed type list, an empty title, and OCR text capped at about 4,000 characters.\n\n## Results (60 held-out rows, exact match)\n\nFill in from this task's README after publishing this run's printed numbers.\n\n## Training\n\nLoRA r=16, alpha 16, 3 epochs, lr 2e-4, batch 2 x grad-accum 4, Unsloth 4-bit, Colab GPU. Train/eval split: seed 42, 140/60 from 200 sampled rows, identical split across every model this task is fine-tuned on.\n\n## Usage\n\nLoad with PEFT on top of `Qwen/Qwen3.5-4B`, or serve locally with an adapter-capable runtime. Caveat: evaluated on 60 rows for one field only. Re-validate on your paperless-ngx document types before production use.\n"

with open("apprentice-qwen35-4b-lora-document-types/README.md", "w") as f:
    f.write(model_card)

# The persist cell above already deleted the live model to free GPU
# memory, so push the SAVED FOLDER (adapter + tokenizer + card), never
# the model object.
from huggingface_hub import HfApi
api = HfApi()
api.create_repo(REPO_ID, private=False, exist_ok=True)
api.upload_folder(folder_path="apprentice-qwen35-4b-lora-document-types", repo_id=REPO_ID)
print(f"pushed -> https://huggingface.co/{REPO_ID}")


## Gemma 4 E4B

`google/gemma-4-E4B-it` -- natively multimodal (text/image/audio in, text out) but loads and fine-tunes as a plain causal LM for this text-only task

In [ ]:
# 10. Load Gemma 4 E4B in 4-bit + LoRA
# google/gemma-4-E4B-it is natively multimodal (text/image/audio in, text out) but loads and fine-tunes as a plain causal LM for this text-only task.
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    "google/gemma-4-E4B-it",
    max_seq_length=4096,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

In [ ]:
# 11. Pre-finetune baseline of the raw SLM on the held-out split
raw_slm_score = evaluate_slm(model, tokenizer, devset)
print(f"Gemma 4 E4B raw (no fine-tune): {raw_slm_score:.2f}")

In [ ]:
# 12. Build chat-format training data and fine-tune (SFT)
from datasets import Dataset
from trl import SFTConfig, SFTTrainer

train_ds = Dataset.from_list([to_text(tokenizer, r) for r in trainset])

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        logging_steps=5,
        output_dir="outputs-apprentice-gemma4-e4b-lora-document-types",
        report_to="none",
    ),
)
trainer.train()

In [ ]:
# 13. Post-finetune eval on the SAME held-out split: the comparison that matters
finetuned_score = evaluate_slm(model, tokenizer, devset)
results["Gemma 4 E4B"] = {"raw": raw_slm_score, "finetuned": finetuned_score}

gpt54_baseline = 78.33  # gpt-5.4-mini, verbatim shipped paperless-gpt prompt (README, 2026-07-10)
gpt54_optimized = 81.67  # same run, GEPA-optimized

print("=" * 52)
print(f"gpt-5.4-mini baseline prompt : {gpt54_baseline:.2f}")
print(f"gpt-5.4-mini GEPA-optimized  : {gpt54_optimized:.2f}")
print(f"Gemma 4 E4B raw               : {raw_slm_score:.2f}")
print(f"Gemma 4 E4B fine-tuned        : {finetuned_score:.2f}")
print("=" * 52)
print("GO criterion: fine-tuned SLM >= gpt-5.4-mini baseline prompt score (78.33)")

In [ ]:
# 14. Persist the adapter (plan: artifacts go to a private HF Hub repo, not Drive),
# then free GPU memory before the next model
model.save_pretrained("apprentice-gemma4-e4b-lora-document-types")
tokenizer.save_pretrained("apprentice-gemma4-e4b-lora-document-types")
print("saved -> apprentice-gemma4-e4b-lora-document-types/")

del model, tokenizer, trainer
free_gpu_memory()

In [ ]:
# 14b. Write the model card and push to your HF Hub repo (same card style as
# the published receipt-extraction adapters).
# Fill in the printed numbers from cell 13 before pushing. Never push a card
# with a number that was not actually printed by this run.
from huggingface_hub import login

login()  # paste a HF token with write access when prompted

HF_USERNAME = "YOUR_HF_USERNAME"
REPO_ID = f"{HF_USERNAME}/apprentice-gemma4-e4b-lora-document-types"

model_card = "---\ndatasets: [anirudh1112/corrected-tobacco-dataset-with-ocr]\nbase_model: google/gemma-4-E4B-it\nlibrary_name: peft\nlicense: apache-2.0\ntags: [lora, unsloth, gemma, document-type-classification, apprentice, paperless-gpt]\n---\n\n# Apprentice Gemma 4 E4B LoRA (document type classification)\n\nLoRA adapter fine-tuned on 140 golden examples from a corrected Tobacco3482 OCR-text dataset to classify one document type from: ADVE, Email, Form, Letter, Memo, News, Note, Report, Resume, Scientific.\n\nThe input prompt is the verbatim shipped document-type prompt from icereed/paperless-gpt, filled with English, the allowed type list, an empty title, and OCR text capped at about 4,000 characters.\n\n## Results (60 held-out rows, exact match)\n\nFill in from this task's README after publishing this run's printed numbers.\n\n## Training\n\nLoRA r=16, alpha 16, 3 epochs, lr 2e-4, batch 2 x grad-accum 4, Unsloth 4-bit, Colab GPU. Train/eval split: seed 42, 140/60 from 200 sampled rows, identical split across every model this task is fine-tuned on.\n\n## Usage\n\nLoad with PEFT on top of `google/gemma-4-E4B-it`, or serve locally with an adapter-capable runtime. Caveat: evaluated on 60 rows for one field only. Re-validate on your paperless-ngx document types before production use.\n"

with open("apprentice-gemma4-e4b-lora-document-types/README.md", "w") as f:
    f.write(model_card)

# The persist cell above already deleted the live model to free GPU
# memory, so push the SAVED FOLDER (adapter + tokenizer + card), never
# the model object.
from huggingface_hub import HfApi
api = HfApi()
api.create_repo(REPO_ID, private=False, exist_ok=True)
api.upload_folder(folder_path="apprentice-gemma4-e4b-lora-document-types", repo_id=REPO_ID)
print(f"pushed -> https://huggingface.co/{REPO_ID}")


## Both, side by side

In [ ]:
# 15. Every model's raw + fine-tuned score, printed together
print("=" * 60)
print(f"{'gpt-5.4-mini (teacher)':22s} baseline: 78.33   GEPA-optimized: 81.67")
for name, scores in results.items():
    print(f"{name:22s} raw: {scores['raw']:6.2f}   fine-tuned: {scores['finetuned']:6.2f}")
print("=" * 60)

## After this run

1. Publish every raw/fine-tuned number exactly as printed -- this repo never carries a number that was not measured.
2. Add rows to this task's README and CASE-STUDY.md results tables for both models.
3. Both base models ship Apache-2.0 -- safe to publish both fine-tuned adapters the same way.
4. If APIs error: Unsloth moves fast. Cross-check against the current official Unsloth notebook for whichever model failed.